# Phase 2 pilot -- benign vs random vs PGD, DiT-B/2 vs UNet-B

Presentation layer for the Phase 2 pilot (spec: `docs/phase2_plan.md`;
pre-registration: `experiments/phase2_pilot/PREREG.md`, git tag `phase2-prereg`).
The heavy compute lives in two scripts -- this notebook only loads their
outputs and renders the comparison. Run order:

1. `scripts/run_phase2_pilot.py --model {SiT-B/2,UNet-B} --part {ab,fid}`
   (driven by `scripts/run_phase2_pilot_all.sh`) -> `ab_results.pt`, `fid_features.pt`.
2. `scripts/analyze_phase2_pilot.py` -> `analysis.json`.

**Why three branches.** The Euler sampler is deterministic given `(z_T, y)`,
so a benign re-seed just reproduces the same output -- a "2 sigma of a re-seed"
baseline has zero spread and is useless. Instead, per shared seed we run three
branches: benign, `z_T + delta_rand` (Rademacher, same L_inf and L2 as
saturated PGD), and `z_T + delta_PGD`. Each metric is gated on
`D_i = (M_PGD - M_ben) - (M_rand - M_ben)`, i.e. how much more PGD shifts it
than an equal-budget random perturbation, with PASS iff `mean(D) >= 2*SE(D)`
and a one-sided Wilcoxon agrees. Go/no-go: at least one of the three metrics
passes. Output damage (LPIPS, latent-L2) is always reported next to the
attention metrics -- the claim is about an attention fingerprint of the
architecture, not "attention robustness".

In [ ]:
import sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "third_party" / "sit"))
import numpy as np, torch, matplotlib.pyplot as plt
import importlib.util
spec = importlib.util.spec_from_file_location("apz", REPO / "scripts/analyze_phase2_pilot.py")
A = importlib.util.module_from_spec(spec); spec.loader.exec_module(A)
PILOT = REPO / "experiments/phase2_pilot"
analysis = json.load(open(PILOT / "analysis.json"))
print("models analyzed:", list(analysis))

## 1. Go/no-go summary + comparison table

In [ ]:
def gate_row(g):
    return f"mean(D)={g['mean']:+.4f}  2*SE={2*g['se']:.4f}  d={g['cohens_d']:+.2f}  Wilcoxon_p={g['wilcoxon_p']:.3g}  {'PASS' if g['pass'] else 'fail'}"

for model, r in analysis.items():
    ab = r["ab_n256"]
    print(f"\n{'='*78}\n{model}   [N=256 locus: {ab['n_layers']} layers, K={ab['k_rand']} random draws]\n{'='*78}")
    print(f"  A  entropy        {gate_row(ab['entropy']['gate'])}")
    print(f"     |shift|: PGD={ab['entropy']['abs_shift_pgd_mean']:.4f}  rand={ab['entropy']['abs_shift_rand_mean']:.4f}")
    for k,label in (("flatness_ratio","B  flatness ratio"),("erank_rv","B  eff.rank (RV) ")):
        print(f"  {label} {gate_row(ab[k]['gate'])}")
        print(f"     drop:    PGD={ab[k]['drop_pgd_mean']:+.4f}  rand={ab[k]['drop_rand_mean']:+.4f}  (benign level={ab[k]['benign_mean']:.4f})")
    if "fid" in r:
        f = r["fid"]
        print(f"  C  Diff-FID       ddFID={f['ddFID']:+.3f}  CI95=[{f['ddFID_ci95'][0]:+.3f}, {f['ddFID_ci95'][1]:+.3f}]  {'PASS' if f['pass'] else 'fail'}")
        print(f"     FID: benign={f['fid_ben']:.2f}  rand={f['fid_rand']:.2f}  PGD={f['fid_pgd']:.2f}  (n={f['n']}, biased at small n)")
    lp = r["lpips"]
    print(f"  LPIPS efficacy   PGD={lp['lpips_pgd_mean']:.4f}  rand={lp['lpips_rand_mean']:.4f}  ratio={lp['lpips_ratio']:.2f}  {'OK' if lp['pass'] else 'FALLBACK -> image-space objective'}")
    print(f"  L2 output damage PGD={lp['l2_out_pgd_mean']:.2f}  rand={lp['l2_out_rand_mean']:.2f}")
    passed = [m for m,ok in [("A/entropy",ab['entropy']['gate']['pass']),
              ("B/flatness",ab['flatness_ratio']['gate']['pass']),
              ("B/erank",ab['erank_rv']['gate']['pass']),
              ("C/diff-fid", r.get('fid',{}).get('pass',False))] if ok]
    print(f"  >>> GO/NO-GO: {'GO - '+', '.join(passed)+' pass' if passed else 'NO-GO - diagnose before Phase 3'}")
    print(f"  >>> PRIMARY METRIC: {r['election']['primary']}  ({r['election']['reason']})")

## 2. Effect-size comparison (DiT vs UNet)

Bars are Cohen's *d* of the PGD-vs-random contrast on the N=256 locus. It is a
per-model aggregate: the N=256 blocks are not depth-matched between the two
models, so this is not a depth-resolved comparison.

In [ ]:
metrics = ["entropy", "flatness_ratio", "erank_rv"]
models = list(analysis)
d = {m: [analysis[m]["ab_n256"][k]["gate"]["cohens_d"] for k in metrics] for m in models}
x = np.arange(len(metrics)); w = 0.38
fig, ax = plt.subplots(figsize=(8,4))
for i,m in enumerate(models):
    ax.bar(x + (i-0.5)*w, d[m], w, label=m)
ax.axhline(0, color="k", lw=0.7); ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel("Cohen's d  (PGD vs random-delta)"); ax.set_title("Attention-metric effect size on the N=256 locus")
ax.legend(); fig.tight_layout(); plt.show()

## 3. Descriptive per-timestep profile (context, not a gate)

Mean over layers/heads of the metric, per Euler step, benign vs PGD vs rand.

In [ ]:
def per_step(ab_attn, branch, metric, token_filter=256):
    st = ab_attn[branch]
    names = [n for n in st if int(st[n]["n_tokens"])==token_filter]
    return torch.stack([st[n][metric].mean(dim=(1,2)) for n in names]).mean(0).numpy()

fig, axes = plt.subplots(len(models), 3, figsize=(13, 3.4*len(models)), squeeze=False)
for i,m in enumerate(models):
    ab = torch.load(PILOT / A.SLUGS[m] / "ab_results.pt", weights_only=False)["attn"]
    for j,metric in enumerate(metrics):
        ax = axes[i][j]
        ax.plot(per_step(ab, "ben", metric), label="benign", lw=2)
        ax.plot(per_step(ab, "pgd", metric), label="PGD", lw=2, ls="--")
        ax.plot(per_step(ab, "rand0", metric), label="rand", lw=1, alpha=0.6)
        ax.set_title(f"{m} - {metric}"); ax.set_xlabel("Euler step (t=0->1)")
        if j==0: ax.set_ylabel("metric (N=256, mean L,H)")
        if i==0 and j==2: ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

## 4. Qualitative: benign vs PGD samples (best/worst/median by L2 damage)

In [ ]:
from diffusers import AutoencoderKL
from src.evaluation.fid import VAE_REPO, SCALING_FACTOR
vae = AutoencoderKL.from_pretrained(VAE_REPO, torch_dtype=torch.float16).to("cuda").eval()
def dec(z):
    with torch.inference_mode():
        img = vae.decode(z.to("cuda").to(torch.float16)/SCALING_FACTOR).sample
    return ((img.float()+1)/2).clamp(0,1).cpu()

for m in models:
    d = torch.load(PILOT / A.SLUGS[m] / "ab_results.pt", weights_only=False)
    l2 = d["l2_out"]["pgd"].numpy()
    order = np.argsort(l2); picks = {"worst":order[-1],"median":order[len(order)//2],"best":order[0]}
    fig, axes = plt.subplots(2, 3, figsize=(9,6.2))
    for c,(label,idx) in enumerate(picks.items()):
        axes[0][c].imshow(dec(d["z0"]["ben"][idx:idx+1])[0].permute(1,2,0)); axes[0][c].set_title(f"benign ({label})")
        axes[1][c].imshow(dec(d["z0"]["pgd"][idx:idx+1])[0].permute(1,2,0)); axes[1][c].set_title(f"PGD  L2={l2[idx]:.1f}")
    for a in axes.flat: a.axis("off")
    fig.suptitle(f"{m}: benign (top) vs PGD (bottom)"); fig.tight_layout(); plt.show()

## 5. Conclusions -> Phase 3

- **Elected primary metric** (section 1) -> axis of the Phase 3 main table.
- **eps calibration**: does 0.05 produce a measurable above-random shift? ->
  whether to re-center the Phase 3 grid {0.01, 0.02, 0.05, 0.1}.
- **LPIPS check** must read OK (ratio >= 1.5); else the image-space-objective
  fallback runs before any null attention claim.
- Phase 3 uses **disjoint seeds** (pilot exploratory / Phase 3 confirmatory).